# Laboratorio B: Red multicapa (MLP) desde cero con NumPy

**Redes Neuronales II - Especializacion en Inteligencia Artificial**
**Autor:** Harold Steven Rueda Lara
**Objetivo:** Reemplazar la neurona unica por una red 3 -> 4 -> 1 (capa oculta con ReLU, salida con Sigmoide) y aplicar backpropagation a traves de varias capas.


## Paso 1-4. Cargar y preparar los mismos datos de frutas del Laboratorio A

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

X = np.array([
    [150, 7.5, 0.10], [160, 8.0, 0.15], [145, 7.2, 0.05],
    [170, 8.3, 0.20], [155, 7.8, 0.10], [165, 8.1, 0.15],
    [140, 7.0, 0.05], [175, 8.5, 0.20], [158, 7.9, 0.10],
    [162, 8.2, 0.15],
    [120, 18.0, 0.90], [125, 19.0, 0.95], [115, 17.5, 0.85],
    [130, 20.0, 1.00], [118, 18.5, 0.90], [123, 19.5, 0.95],
    [110, 17.0, 0.80], [128, 20.5, 1.00], [121, 18.8, 0.90],
    [126, 19.2, 0.95]
], dtype=float)
y = np.array([0]*10 + [1]*10).reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def calcular_error(y_real, y_predicha):
    epsilon = 1e-8
    return -np.mean(
        y_real * np.log(y_predicha + epsilon)
        + (1 - y_real) * np.log(1 - y_predicha + epsilon)
    )
print("Datos listos:", X_train.shape, X_test.shape)

Datos listos: (16, 3) (4, 3)


## Paso 11. Funciones de activacion (ReLU para la capa oculta)

In [ ]:
def relu(z):
    return np.maximum(0, z)

def derivada_relu(z):
    return (z > 0).astype(float)

## Paso 12. Inicializar parametros (arquitectura 3 -> 4 -> 1)
Usamos inicializacion He (`sqrt(2/n)`), recomendada cuando la capa usa ReLU.

In [2]:
np.random.seed(42)
W1 = np.random.randn(3, 4) * np.sqrt(2 / 3)
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * np.sqrt(2 / 4)
b2 = np.zeros((1, 1))
print("W1:", W1.shape)
print("W2:", W2.shape)

W1: (3, 4)
W2: (4, 1)


## Paso 13-14. Forward propagation y backpropagation multicapa
La senal de correccion nace en la salida (`dZ2`) y se propaga hacia la capa oculta (`dZ1`) aplicando la regla de la cadena a traves de `W2` y de la derivada de ReLU.

In [ ]:
Z1 = X_train @ W1 + b1
A1 = relu(Z1)
Z2 = A1 @ W2 + b2
A2 = sigmoid(Z2)

m = len(X_train)
dZ2 = A2 - y_train
dW2 = A1.T @ dZ2 / m
db2 = np.mean(dZ2, axis=0, keepdims=True)

dA1 = dZ2 @ W2.T
dZ1 = dA1 * derivada_relu(Z1)
dW1 = X_train.T @ dZ1 / m
db1 = np.mean(dZ1, axis=0, keepdims=True)

print("Forma dW1:", dW1.shape, "| Forma dW2:", dW2.shape)

## Paso 15. Entrenamiento completo (3000 epocas)

In [3]:
np.random.seed(42)
W1 = np.random.randn(3, 4) * np.sqrt(2 / 3)
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * np.sqrt(2 / 4)
b2 = np.zeros((1, 1))
learning_rate = 0.05
epochs = 3000
historial_loss = []

for epoch in range(epochs):
    Z1 = X_train @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)

    error = calcular_error(y_train, A2)
    historial_loss.append(error)

    m = len(X_train)
    dZ2 = A2 - y_train
    dW2 = A1.T @ dZ2 / m
    db2 = np.mean(dZ2, axis=0, keepdims=True)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * derivada_relu(Z1)
    dW1 = X_train.T @ dZ1 / m
    db1 = np.mean(dZ1, axis=0, keepdims=True)

    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    if epoch % 500 == 0:
        print("Epoca:", epoch, "- Error:", round(error, 4))

Epoca: 0 - Error: 0.9529
Epoca: 500 - Error: 0.0555
Epoca: 1000 - Error: 0.0259
Epoca: 1500 - Error: 0.0166
Epoca: 2000 - Error: 0.0122
Epoca: 2500 - Error: 0.0096


## Grafica de la curva de perdida

In [ ]:
plt.plot(historial_loss)
plt.title("Laboratorio B - Red multicapa 3->4->1 (Frutas)")
plt.xlabel("Epoca")
plt.ylabel("Binary Cross-Entropy")
plt.grid(alpha=0.3)
plt.show()

## Paso 16. Evaluar la red multicapa

In [4]:
Z1_test = X_test @ W1 + b1
A1_test = relu(Z1_test)
Z2_test = A1_test @ W2 + b2
probabilidades_test = sigmoid(Z2_test)
predicciones_multicapa = (probabilidades_test >= 0.5).astype(int)
accuracy_multicapa = np.mean(predicciones_multicapa == y_test)
print("Predicciones:", predicciones_multicapa.ravel())
print("Reales:", y_test.ravel())
print("Accuracy:", accuracy_multicapa)

Predicciones: [0 0 1 1]
Reales: [0 0 1 1]
Accuracy: 1.0


## Conclusion del laboratorio
La red multicapa (3 -> 4 -> 1) tambien alcanzo **accuracy = 1.0**, y su curva de perdida muestra una convergencia mas gradual que la neurona unica, propia de tener mas parametros que ajustar via backpropagation.